# Preprocess Amazon Grocery and Gourmet Food for ICSRec

**Course:** CCAI 422 Recommender Systems — University of Jeddah, Spring 2025/2026

**Paper:** Qin et al., *Intent Contrastive Learning with Cross Subsequences for Sequential Recommendation*, WSDM 2024.

**Goal of this notebook:** Convert the McAuley Lab Amazon Reviews 2023 *Grocery and Gourmet Food* dataset (5-core filtered) into the exact input format that the ICSRec codebase expects:

```
user_id item1 item2 item3 ...   (one user per line, items in chronological order)
```

### Preprocessing steps in this notebook
1. **Load** the McAuley Lab 5-core deduplicated CSV from Hugging Face
2. **Inspect** the raw data and handle missing values
3. **Sort** each user's interactions by timestamp (critical for sequential rec)
4. **Group** interactions into per-user sequences
5. **Filter** sequences that are too short (ICSRec needs ≥ 3 interactions per user for train/valid/test)
6. **Remap** user and item IDs to consecutive integers starting from 1 (ICSRec convention — id 0 is reserved for padding)
7. **Save** in ICSRec format and copy to Google Drive
8. **Report statistics** for the project report (# users, # items, # interactions, sparsity, avg seq length)

## Step 0: Install dependencies and mount Drive

We only need `pandas`, `numpy`, and `datasets` (Hugging Face). `pandas` and `numpy` come with Colab; `datasets` we install.

In [1]:
!pip install -q datasets

from google.colab import drive
drive.mount('/content/drive')

# Output directory on your Google Drive (change if you want).
# We'll save the processed file here so you don't lose it between sessions.
import os
DRIVE_DIR = '/content/drive/MyDrive/ICSRec_project'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive output dir:', DRIVE_DIR)

Mounted at /content/drive
Drive output dir: /content/drive/MyDrive/ICSRec_project


## Step 1: Load the 5-core dataset from Hugging Face

McAuley Lab provides a pre-deduplicated, 5-core filtered CSV at:
`McAuley-Lab/Amazon-Reviews-2023` → `5core_rating_only_Grocery_and_Gourmet_Food`

Columns: `user_id` (string hash), `parent_asin` (item id, string), `rating` (1–5 float), `timestamp` (Unix milliseconds).

*5-core* means every remaining user has ≥ 5 interactions and every remaining item has ≥ 5 interactions — already done for us.

In [12]:
# Direct download of the CSV file from Hugging Face Hub.
# We bypass load_dataset() entirely because the new datasets library blocks
# loading scripts for security reasons.
from huggingface_hub import hf_hub_download
import pandas as pd

csv_path = hf_hub_download(
    repo_id='McAuley-Lab/Amazon-Reviews-2023',
    filename='benchmark/5core/rating_only/Grocery_and_Gourmet_Food.csv',
    repo_type='dataset'
)
print('Downloaded to:', csv_path)

# The CSV has columns: user_id, parent_asin, rating, timestamp
df = pd.read_csv(csv_path)
print('\nShape:', df.shape)
print('\nFirst rows:')
df.head()

Downloaded to: /root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e/benchmark/5core/rating_only/Grocery_and_Gourmet_Food.csv

Shape: (3948741, 4)

First rows:


,user_id,parent_asin,rating,timestamp
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B004OT1TA8,5.0,1441260352000
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B01NAYX4S3,5.0,1523093131195
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B08KQLCRG7,5.0,1534441227514
3,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B0C5KQVWJL,5.0,1548207240942
4,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B09JP7Q6W8,5.0,1548207988441


## Step 2: Inspect and handle missing values

Even on a 5-core filtered dataset, we double-check for nulls and duplicates.

In [13]:
print('Column types:')
print(df.dtypes)

print('\nNull counts per column:')
print(df.isnull().sum())

print('\nNumber of duplicate (user, item, timestamp) rows:',
      df.duplicated(subset=['user_id', 'parent_asin', 'timestamp']).sum())

# Drop any rows with missing critical fields (defensive — should be zero)
df = df.dropna(subset=['user_id', 'parent_asin', 'timestamp'])

# Drop exact duplicate interactions (a user buying the same item at the same second)
df = df.drop_duplicates(subset=['user_id', 'parent_asin', 'timestamp'])

print('\nAfter cleaning, shape:', df.shape)

Column types:
user_id         object
parent_asin     object
rating         float64
timestamp        int64
dtype: object

Null counts per column:
user_id        0
parent_asin    0
rating         0
timestamp      0
dtype: int64

Number of duplicate (user, item, timestamp) rows: 0

After cleaning, shape: (3948741, 4)


In [14]:
# ============================================================
# Step 2.5: Iterative k-core filtering
# ------------------------------------------------------------
# Purpose: bring the dataset to a size comparable to the paper's
# Beauty/Toys/Sports datasets (~200K-500K interactions) for two
# reasons:
#   1) Direct comparability with the original ICSRec experiments
#   2) Faster training on free Colab, enabling more hyperparameter
#      tuning runs in the same wall-clock budget
#
# How it works: repeatedly drop users with < K interactions and
# items with < K interactions until the data is stable (one round
# changes nothing).
# ============================================================

K_CORE = 15   # try 10 first; raise to 15 or 20 if still too big

print(f'Before k-core filtering: {len(df):,} interactions, '
      f'{df["user_id"].nunique():,} users, '
      f'{df["parent_asin"].nunique():,} items')

iteration = 0
while True:
    iteration += 1
    before = len(df)

    # Drop users with too few interactions
    user_counts = df['user_id'].value_counts()
    valid_users = user_counts[user_counts >= K_CORE].index
    df = df[df['user_id'].isin(valid_users)]

    # Drop items with too few interactions
    item_counts = df['parent_asin'].value_counts()
    valid_items = item_counts[item_counts >= K_CORE].index
    df = df[df['parent_asin'].isin(valid_items)]

    after = len(df)
    print(f'  iter {iteration}: {before:,} -> {after:,} interactions')

    # Stop when a full pass changes nothing
    if before == after:
        break

print(f'\nAfter {K_CORE}-core filtering: {len(df):,} interactions, '
      f'{df["user_id"].nunique():,} users, '
      f'{df["parent_asin"].nunique():,} items')

Before k-core filtering: 3,948,741 interactions, 404,760 users, 132,865 items
  iter 1: 3,948,741 -> 893,371 interactions
  iter 2: 893,371 -> 515,699 interactions
  iter 3: 515,699 -> 409,754 interactions
  iter 4: 409,754 -> 367,459 interactions
  iter 5: 367,459 -> 348,449 interactions
  iter 6: 348,449 -> 339,289 interactions
  iter 7: 339,289 -> 334,356 interactions
  iter 8: 334,356 -> 331,776 interactions
  iter 9: 331,776 -> 330,713 interactions
  iter 10: 330,713 -> 330,013 interactions
  iter 11: 330,013 -> 329,607 interactions
  iter 12: 329,607 -> 329,453 interactions
  iter 13: 329,453 -> 329,411 interactions
  iter 14: 329,411 -> 329,411 interactions

After 15-core filtering: 329,411 interactions, 12,713 users, 8,766 items


## Step 3: Sort each user's interactions by timestamp

This is **the** critical step for sequential recommendation. ICSRec models the order of items a user interacted with, so the timestamps must be respected exactly.

In [15]:
# Sort first by user, then by timestamp ascending (oldest interaction first)
df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
print('Sorted. First 10 rows:')
df.head(10)

Sorted. First 10 rows:


,user_id,parent_asin,rating,timestamp
0,AE22KGLDL2HSG3G77565OO6HGYRA,B001GB7IJI,5.0,1462086370000
1,AE22KGLDL2HSG3G77565OO6HGYRA,B01F3PI0BW,5.0,1462087369000
2,AE22KGLDL2HSG3G77565OO6HGYRA,B004H6MUDI,5.0,1462087400000
3,AE22KGLDL2HSG3G77565OO6HGYRA,B00Z9YWAO6,5.0,1462087695000
4,AE22KGLDL2HSG3G77565OO6HGYRA,B0C7JLM52P,5.0,1462087797000
5,AE22KGLDL2HSG3G77565OO6HGYRA,B00VC5CUAU,5.0,1462088115000
6,AE22KGLDL2HSG3G77565OO6HGYRA,B00347UJQQ,5.0,1462088502000
7,AE22KGLDL2HSG3G77565OO6HGYRA,B09PGB16PF,5.0,1462088523000
8,AE22KGLDL2HSG3G77565OO6HGYRA,B00CFVDIKG,5.0,1462088592000
9,AE22KGLDL2HSG3G77565OO6HGYRA,B004J0M22Q,5.0,1462088982000


## Step 4: Group into per-user item sequences

In [16]:
# For each user, collect the chronological list of items they interacted with.
# `sort=False` keeps the row order we already created in step 3.
user_sequences = df.groupby('user_id', sort=False)['parent_asin'].apply(list)

print('Number of unique users:', len(user_sequences))
print('Example user, first 10 items in their sequence:')
print(user_sequences.iloc[0][:10])

Number of unique users: 12713
Example user, first 10 items in their sequence:
['B001GB7IJI', 'B01F3PI0BW', 'B004H6MUDI', 'B00Z9YWAO6', 'B0C7JLM52P', 'B00VC5CUAU', 'B00347UJQQ', 'B09PGB16PF', 'B00CFVDIKG', 'B004J0M22Q']


## Step 5: Filter users with too-short sequences

ICSRec uses a **leave-one-out** evaluation strategy: the last item of each user goes to the test set, the second-to-last to the validation set, and the rest is training data. This means we need **at least 3 interactions per user**.

The 5-core filter already guarantees ≥ 5, but we keep this safety check.

In [17]:
MIN_SEQ_LEN = 3  # need at least 3: train + valid + test
user_sequences = user_sequences[user_sequences.apply(len) >= MIN_SEQ_LEN]
print(f'Users with sequence length >= {MIN_SEQ_LEN}: {len(user_sequences)}')

Users with sequence length >= 3: 12713


## Step 6: Remap user and item IDs to consecutive integers

ICSRec expects:
- User IDs starting from 1 (the leading number on each line)
- Item IDs starting from 1 (id 0 is reserved for sequence padding inside the model)

The raw IDs are strings like `AHFZ4...` — we need to convert them to integers and remember the mapping in case we want to recover the original IDs later.

In [18]:
# Build user mapping (string -> int starting at 1)
user2id = {u: i + 1 for i, u in enumerate(user_sequences.index)}

# Build item mapping — only over items that appear in the kept users
all_items_in_kept_users = set()
for seq in user_sequences:
    all_items_in_kept_users.update(seq)
item2id = {it: i + 1 for i, it in enumerate(sorted(all_items_in_kept_users))}

print(f'Number of users: {len(user2id)}')
print(f'Number of items: {len(item2id)}')

# Apply the mapping to build the final integer-id sequences
remapped = {}
for user_str, seq in user_sequences.items():
    remapped[user2id[user_str]] = [item2id[it] for it in seq]

print('\nExample remapped sequence (user 1, first 10 items):')
print(remapped[1][:10])

Number of users: 12713
Number of items: 8766

Example remapped sequence (user 1, first 10 items):
[576, 2336, 842, 2077, 8465, 1996, 705, 5949, 1253, 848]


## Step 7: Write the ICSRec input file

Format: one user per line, space-separated, leading integer is the user ID. This matches `data/Beauty.txt` in the ICSRec repo exactly.

In [19]:
OUT_NAME = 'Grocery_and_Gourmet_Food.txt'
out_path = os.path.join(DRIVE_DIR, OUT_NAME)

with open(out_path, 'w') as f:
    for uid in sorted(remapped.keys()):
        line = str(uid) + ' ' + ' '.join(str(it) for it in remapped[uid])
        f.write(line + '\n')

print('Saved to:', out_path)

# Quick sanity check — first three lines
print('\nFirst 3 lines of the output file:')
with open(out_path) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line[:120] + ('...' if len(line) > 120 else ''))

Saved to: /content/drive/MyDrive/ICSRec_project/Grocery_and_Gourmet_Food.txt

First 3 lines of the output file:
1 576 2336 842 2077 8465 1996 705 5949 1253 848 1838 540 4587 1493 6732 8191 1995 7299 1464 8403

2 1685 8397 7700 634 5159 3913 696 7703 3653 3245 4389 1655 7843 6474 7677 3727 1799

3 6937 4240 4023 4705 6552 1095 5135 5369 5956 5735 6553 5723 7693 7461 8105 5897 1992 4181 7202 8288 5856 8383 5843 198...


## Step 8: Dataset statistics for the project report

In [20]:
import numpy as np

n_users = len(remapped)
n_items = len(item2id)
seq_lengths = np.array([len(s) for s in remapped.values()])
n_interactions = int(seq_lengths.sum())
sparsity = 1.0 - (n_interactions / (n_users * n_items))

print('========================================')
print('  Grocery and Gourmet Food (5-core)')
print('  Statistics for the project report')
print('========================================')
print(f'  # users:           {n_users:>10,}')
print(f'  # items:           {n_items:>10,}')
print(f'  # interactions:    {n_interactions:>10,}')
print(f'  avg seq length:    {seq_lengths.mean():>10.2f}')
print(f'  median seq length: {int(np.median(seq_lengths)):>10}')
print(f'  max seq length:    {int(seq_lengths.max()):>10}')
print(f'  min seq length:    {int(seq_lengths.min()):>10}')
print(f'  sparsity:          {sparsity*100:>10.4f} %')

  Grocery and Gourmet Food (5-core)
  Statistics for the project report
  # users:               12,713
  # items:                8,766
  # interactions:       329,411
  avg seq length:         25.91
  median seq length:         20
  max seq length:           473
  min seq length:            15
  sparsity:             99.7044 %


## Step 9: Save the ID mappings

In [21]:
import json

with open(os.path.join(DRIVE_DIR, 'user2id.json'), 'w') as f:
    json.dump(user2id, f)
with open(os.path.join(DRIVE_DIR, 'item2id.json'), 'w') as f:
    json.dump(item2id, f)

print('Saved id mappings to', DRIVE_DIR)
print('Files in your Drive output folder:')
for f in sorted(os.listdir(DRIVE_DIR)):
    size_mb = os.path.getsize(os.path.join(DRIVE_DIR, f)) / 1e6
    print(f'  {f:<40} {size_mb:>8.2f} MB')

Saved id mappings to /content/drive/MyDrive/ICSRec_project
Files in your Drive output folder:
  Grocery_and_Gourmet_Food.txt                 1.68 MB
  item2id.json                                 0.17 MB
  user2id.json                                 0.48 MB
